# 🇧🇷 Brasil Online — Rodar no Colab

MMORPG de mundo aberto (protótipo frontend, Etapa 1).

**Como usar:** clique na célula abaixo e aperte ▶ (ou `Shift+Enter`).
Depois clique no link **"Abrir jogo"** que aparecer.

> Se o repositório for **privado**, preencha `GITHUB_TOKEN` na célula.
> Deixe a célula em execução enquanto joga.

In [ ]:
import os, subprocess, threading, http.server, socketserver, functools

# ===== CONFIGURAÇÃO =====
REPO_URL = "https://github.com/wallafsilva03-spec/Jogo-RP-mundo-aberto-.git"
BRANCH   = "claude/brasil-online-mmorpg-ekazrg"
DEST     = "/content/brasil-online"
PORT     = 8000
GITHUB_TOKEN = ""  # se o repo for privado, cole um token (escopo: repo)

# ===== 1) Clonar / atualizar =====
url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if os.path.isdir(os.path.join(DEST, ".git")):
    print("📁 Atualizando repositório...")
    subprocess.run(["git", "-C", DEST, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    print("⬇️  Clonando o jogo...")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", url, DEST], check=True)
print("✅ Arquivos prontos.")

# ===== 2) Servidor HTTP estático =====
class Handler(http.server.SimpleHTTPRequestHandler):
    def end_headers(self):
        self.send_header("Cache-Control", "no-store")
        super().end_headers()
    def log_message(self, *a):
        pass

socketserver.TCPServer.allow_reuse_address = True
httpd = socketserver.TCPServer(("0.0.0.0", PORT), functools.partial(Handler, directory=DEST))
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"🚀 Servidor rodando na porta {PORT}")

# ===== 3) Link público (proxy do Colab) =====
print("\n============================================")
print("  ✅ BRASIL ONLINE PRONTO! Clique no link abaixo:")
print("============================================\n")
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(PORT, path="/index.html")
print("\n💡 Deixe esta célula em execução enquanto joga. Bom jogo!")